In [ ]:
import os
import pandas as pd
import numpy as np

PROCESSED_DIR = os.path.join("..", "data", "processed")
REPORTS_DIR = os.path.join("..", "reports")

In [ ]:
# Load recommendations and portfolio weights
rec_df = pd.read_csv(os.path.join(REPORTS_DIR, "recommendations.csv"))
weights_df = pd.read_csv(os.path.join(REPORTS_DIR, "portfolio_weights.csv"))

# Merge target weights with asset recommendations
df = rec_df.merge(weights_df, on="Ticker", how="inner")

In [ ]:
PORTFOLIO_VALUE = 100000.0  # Total portfolio target in USD

# Fetch latest prices (or extract from latest processed dataset)
latest_prices = {"AAPL": 180.50, "MSFT": 420.10, "GOOGL": 175.25, "AMZN": 185.00}
df["Current Price ($)"] = df["Ticker"].map(latest_prices)

# Calculate dollar allocation and target share quantities
df["Target Allocation ($)"] = df["Weight"] * PORTFOLIO_VALUE
df["Target Shares"] = (df["Target Allocation ($)"] / df["Current Price ($)"]).astype(int)

# Determine order type based on signal
df["Order Type"] = df["Recommendation Signal"].apply(
    lambda x: "BUY" if "BUY" in x else ("SELL" if "SELL" in x else "HOLD")
)

In [ ]:
rebalancing_orders = df[[
    "Ticker", "Current Price ($)", "Weight", 
    "Target Allocation ($)", "Target Shares", "Order Type"
]]

rebalancing_orders.columns = [
    "Ticker", "Price ($)", "Optimal Weight", 
    "Target Value ($)", "Shares to Trade", "Action"
]

rebalancing_orders

In [ ]:
output_path = os.path.join(REPORTS_DIR, "rebalancing_orders.csv")
rebalancing_orders.to_csv(output_path, index=False)
print(f"Successfully generated rebalancing orders and saved to {output_path}")